## SILICON LABS: OPTION-IMPLIED DEAL PROBABILITIES

Texas Instruments agreed on 2026-02-04 to buy Silicon Labs (SLAB) for \$231.00 per
share in cash. This notebook follows the plan in `report/report.typ`: start from the
stock-only estimate of the completion probability, then pull SLAB's option chain and
check that it has enough strikes below the offer to fit the mixture model.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ----------------------------------------------------------------------------
# Imports
# ----------------------------------------------------------------------------

# Standard library imports
from datetime import UTC, datetime, time, timedelta
from pathlib import Path

# Third-party imports
import polars as pl

# First-party imports
from xpectral.data import DatabentoREST, MassiveREST

## Deal terms

From the merger announcement. The outside date and the regulatory milestones (antitrust
filings, shareholder vote) come from the merger agreement and are still to be added.

In [ ]:
ticker = "SLAB"
acquirer = "TXN"
W = 231.00  # total workout: cash per SLAB share at closing
announced = datetime(2026, 2, 4, tzinfo=UTC)
expected_close = "H1 2027"  # TODO: outside date from the merger agreement

## Stock prices

Daily SLAB bars from a month before the announcement. The last close before the
announcement is the usual proxy for the fallback price $S_\text{break}$, and the latest
close is $S_0$.

In [ ]:
bars = (
    MassiveREST()
    .get_aggregate_bars(
        [ticker],
        1,
        "day",
        announced - timedelta(days=30),
        datetime.now(UTC),
    )
    .collect()
    .sort("timestamp")
)

S_pre = bars.filter(pl.col("timestamp").dt.date() < announced.date())["close"][-1]
S_0 = bars["close"][-1]
print(f"last close before announcement: {S_pre:.2f}")
print(f"latest close ({bars['timestamp'][-1]:%Y-%m-%d}): {S_0:.2f}")
print(f"spread to W: {W - S_0:.2f} ({W / S_0 - 1:.2%})")

## Stock-only baseline

Report equation (1), ignoring discounting and dividends:

$$p = \frac{S_0 - S_\text{break}}{W - S_\text{break}}.$$

The fallback is not observed, so the table shows how $p$ moves with the guess, from 20%
below to 20% above the pre-announcement close. The market has moved since February, so
the unadjusted close is only a starting point.

In [ ]:
baseline = pl.DataFrame({"S_break": [S_pre * k for k in (0.8, 0.9, 1.0, 1.1, 1.2)]})
baseline = baseline.with_columns(p=(S_0 - pl.col("S_break")) / (W - pl.col("S_break")))
baseline

## Option quotes

One trading day of consolidated best bid and offer, sampled each minute, for every SLAB
option. Databento bills by data volume, so check the cost first. The cost check is free.

In [ ]:
day = datetime(2026, 10, 2, tzinfo=UTC)
databento = DatabentoREST()
cost = databento.get_option_quotes_cost([ticker], day, day + timedelta(days=1))
print(f"cost: ${cost:,.2f}")

Run this once the cost is acceptable. The pull is cached in `data/`, so later runs read
the file instead of paying again.

In [ ]:
quotes_path = Path(f"data/slab-options-cbbo-1m-{day:%Y-%m-%d}.parquet")
if not quotes_path.exists():
    quotes_path.parent.mkdir(exist_ok=True)
    quotes = databento.get_option_quotes([ticker], day, day + timedelta(days=1))
    quotes.collect().write_parquet(quotes_path)
quotes = pl.read_parquet(quotes_path)
quotes.head()

## End-of-day chain

Each contract's last quote at or before the 4 pm close, keeping only quotes with a
positive bid.

In [ ]:
chain = (
    quotes.filter(pl.col("ts_recv").dt.time() <= time(16, 0))
    .sort("ts_recv")
    .group_by("symbol")
    .last()
    .filter(pl.col("bid_px_00") > 0)
    .with_columns(mid=(pl.col("bid_px_00") + pl.col("ask_px_00")) / 2)
    .sort("expiration", "option_type", "strike")
)
chain.select("expiration", "option_type", "strike", "bid_px_00", "ask_px_00", "mid")

## Chain quality

Puts struck below $W$ pay only if the deal breaks (report equation 3), and the fit needs
at least three of them per expiry with a positive bid. This lists, for each expiry, how
many such puts exist and at which strikes.

In [ ]:
(
    chain.filter((pl.col("option_type") == "P") & (pl.col("strike") < W))
    .group_by("expiration")
    .agg(n_puts=pl.len(), strikes=pl.col("strike").sort())
    .sort("expiration")
)